# EFM3D / EVL — Deep Dive Demo Notebook

This notebook walks through the **Egocentric Voxel Lifting (EVL)** architecture from the EFM3D paper (arXiv:2406.10224). We will:

1. **Understand the architecture** — build each component step by step
2. **Construct synthetic Aria-like data** — create fake batch tensors matching the real data format
3. **Run a full forward pass** — trace tensor shapes through every stage
4. **Visualize** — voxel grids, occupancy predictions, OBB detection outputs
5. **Run real inference** (if checkpoint is available)

---

## Architecture Overview

```
Aria Glasses Input
  |-- RGB fisheye camera (1408x1408)
  |-- SLAM-L camera (640x480)           [optional]
  |-- SLAM-R camera (640x480)           [optional]
  |-- Semi-dense point cloud (per frame)
  +-- Camera poses (T_world_rig per frame)
        |
        v
  +-----------------------------+
  |  2D Backbone (DINOv2 ViT-B) |  <- frozen pretrained
  |  768-dim feature tokens      |
  +-------------+---------------+
                |
                v
  +-----------------------------+
  |  DPT Head (2D upsampling)    |  <- learnable
  |  768 -> 64 channels          |
  +-------------+---------------+
                |
                v
  +-----------------------------+
  |  Lifter (2D -> 3D)           |
  |  * Create gravity-aligned    |
  |    voxel grid (96^3)         |
  |  * Project voxel centers     |
  |    into each camera          |
  |  * Sample 2D features        |
  |  * Aggregate across T frames |
  |  * Concat point mask + free  |
  |  Output: Bx66x96x96x96      |
  +-------------+---------------+
                |
                v
  +-----------------------------+
  |  3D Neck (InvResNet-FPN)     |  <- learnable
  |  Encoder-decoder with skip   |
  |  connections                  |
  |  66->64->96->128->160 (enc)  |
  |  FPN upsampling back to 64   |
  +--------+--------+-----------+
           |        |
           v        v
  +----------+ +--------------------+
  | Occ Head | | OBB Detection Head |
  | -> 1 ch  | | |-- Centerness (1) |
  | sigmoid  | | |-- BBox size (7)  |
  |          | | +-- Class (29)     |
  +----------+ +--------------------+
       |              |
       v              v
  Surface mesh    3D OBB predictions
  (via TSDF        (via NMS + tracking)
   fusion)
```

## 0. Setup & Imports

In [ ]:
import sys, os

# Make sure we're in the EFM3D repo root
EFM3D_DIR = "/research/repos/efm3d"
os.chdir(EFM3D_DIR)
if EFM3D_DIR not in sys.path:
    sys.path.insert(0, EFM3D_DIR)

import torch
import numpy as np
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import hydra

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Understanding the Config System

EVL uses Hydra/OmegaConf configs. Let's examine both the **server** (96^3 voxels) and **desktop** (48^3 voxels) configurations.

In [ ]:
# Load and compare both configs
cfg_server = OmegaConf.load("efm3d/config/evl_inf.yaml")
cfg_desktop = OmegaConf.load("efm3d/config/evl_inf_desktop.yaml")

print("=== Server Config (needs ~20GB VRAM) ===")
print(f"  Voxel grid: {cfg_server.video_backbone3d.voxel_size} = {96**3:,} voxels")
print(f"  Voxel extent: {cfg_server.video_backbone3d.voxel_extent} meters")
print(f"  Feature dim: in={cfg_server.video_backbone3d.in_dim} -> out={cfg_server.video_backbone3d.out_dim}")
print(f"  Neck dims: {cfg_server.neck_hidden_dims}")
print(f"  Head type: {cfg_server.video_backbone3d.head_type}")
print(f"  Streams: {cfg_server.video_backbone3d.streams}")
print()
print("=== Desktop Config (needs ~10GB VRAM) ===")
print(f"  Voxel grid: {cfg_desktop.video_backbone3d.voxel_size} = {48**3:,} voxels")
print(f"  Voxel extent: {cfg_desktop.video_backbone3d.voxel_extent} meters")
print(f"  Feature dim: in={cfg_desktop.video_backbone3d.in_dim} -> out={cfg_desktop.video_backbone3d.out_dim}")
print(f"  Neck dims: {cfg_desktop.neck_hidden_dims}")
print()
print("=== Shared ===")
print(f"  2D backbone: DINOv2 ViT-B/14 (frozen={cfg_server.video_backbone.freeze_encoder})")
print(f"  DINOv2 output dim: {cfg_server.video_backbone.image_tokenizer.dim_out}")
print(f"  Taxonomy: {cfg_server.taxonomy_file}")

In [ ]:
# Load the 29-class taxonomy
import pandas as pd

taxonomy = pd.read_csv("efm3d/config/taxonomy/ase_sem_name_to_id.csv")
print(f"EFM3D detects {len(taxonomy)} object classes:")
for _, row in taxonomy.iterrows():
    print(f"  [{row.sem_id:2d}] {row.sem_name}")

## 2. Build the Model from Config

We use the **desktop config** (48^3 voxels) to keep memory reasonable for exploration.

In [ ]:
# Build EVL model from config (with random weights -- no checkpoint needed)
cfg = OmegaConf.load("efm3d/config/evl_inf_desktop.yaml")
model = hydra.utils.instantiate(cfg)
model.eval()
print(f"\nModel type: {type(model).__name__}")

In [ ]:
# Count parameters by component
def count_params(module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

components = {
    "DINOv2 Backbone (frozen)": model.backbone2d,
    "DPT Head (2D->2D)": model.backbone3d.head,
    "3D Neck (InvResNet-FPN)": model.neck,
    "Occupancy Head": model.occ_head,
    "Centerness Head": model.cent_head,
    "BBox Head": model.bbox_head,
    "Class Head": model.clas_head,
}

print(f"{'Component':<30} {'Total Params':>15} {'Trainable':>15}")
print("-" * 62)
grand_total, grand_trainable = 0, 0
for name, mod in components.items():
    total, trainable = count_params(mod)
    grand_total += total
    grand_trainable += trainable
    print(f"{name:<30} {total:>15,} {trainable:>15,}")
print("-" * 62)
total_all, train_all = count_params(model)
print(f"{'TOTAL':<30} {total_all:>15,} {train_all:>15,}")
print(f"\nDINOv2 accounts for {count_params(model.backbone2d)[0]/total_all*100:.1f}% of total params but is frozen.")
print(f"Effective trainable parameters: {train_all:,}")

## 3. Understanding the Aria Data Format

The EVL model expects a **batch dictionary** with specific keys from Aria's sensor suite. Let's build a synthetic batch to understand the format.

### Key Concepts:
- **Snippet**: A short video clip (e.g. 1 second = 10 frames at 10 Hz)
- **Rig**: The Aria glasses body -- cameras and sensors are mounted on it
- **T_world_snippet**: Transform placing the snippet's reference frame in world coords
- **T_snippet_rig**: Per-frame poses of the rig in snippet coords
- **CameraTW**: Custom tensor-wrapper for camera calibration (fisheye624 model)
- **PoseTW**: Custom tensor-wrapper for SE(3) poses (stored as 3x4 matrix = 12 floats)

In [ ]:
from efm3d.aria.camera import CameraTW, get_aria_camera, RGB_PARAMS, SLAM_PARAMS
from efm3d.aria.pose import PoseTW
from efm3d.aria.aria_constants import (
    ARIA_IMG, ARIA_CALIB, ARIA_IMG_T_SNIPPET_RIG,
    ARIA_SNIPPET_T_WORLD_SNIPPET, ARIA_POINTS_WORLD,
)

# ------- Configuration -------
B = 1          # batch size
T = 10         # frames per snippet (1 sec at 10 Hz)
RGB_H = 700    # Using resolution scale 11 (divisible by 14 for ViT)
RGB_W = 700
N_POINTS = 500 # semi-dense points per frame

print("=== Synthetic Batch Structure ===")
print(f"Batch size: {B}, Frames/snippet: {T}")
print(f"RGB resolution: {RGB_W}x{RGB_H} (scale 11, divisible by 14)")
print(f"Points per frame: {N_POINTS}")

In [ ]:
def create_synthetic_batch(B=1, T=10, rgb_hw=(700, 700), n_points=500, device="cpu"):
    """
    Create a synthetic batch dict mimicking Aria sensor data.
    This lets us run the full model without real data.
    """
    from efm3d.aria.pose import IdentityPose
    H, W = rgb_hw
    batch = {}
    
    # --- RGB images: B x T x 3 x H x W ---
    # Aria RGB is 1408x1408 fisheye, but we use smaller for demo
    batch[ARIA_IMG[0]] = torch.randn(B, T, 3, H, W, device=device)
    
    # --- Camera calibration: CameraTW ---
    # Real Aria uses fisheye624 model with 15 params (f, cx, cy, k0-k5, p0-p1, s0-s3)
    # We scale the default params to match our resolution
    scale = H / 1408.0  # scale from full res
    params = RGB_PARAMS.copy()
    params[0] *= scale  # focal length
    params[1] *= scale  # cx
    params[2] *= scale  # cy
    cam = get_aria_camera(
        params=np.float32(params),
        width=W, height=H,
        valid_radius=int(W * 0.54),  # ~54% of width for fisheye valid region
    )
    # Expand to B x T
    cam = cam.unsqueeze(0).unsqueeze(0).repeat(B, T, 1)
    batch[ARIA_CALIB[0]] = cam.to(device)
    
    # --- Snippet-level world transform: B x 12 ---
    # Identity pose = snippet is at world origin
    # PoseTW stores SE(3) as flattened 3x4 matrix = 12 floats: [R(9) | t(3)]
    id_data = IdentityPose.clone().unsqueeze(0).repeat(B, 1).to(device)
    T_ws = PoseTW(id_data)
    batch[ARIA_SNIPPET_T_WORLD_SNIPPET] = T_ws
    
    # --- Per-frame rig poses: B x T x 12 ---
    # Simulate gentle forward motion (like walking through a room)
    poses = []
    for t in range(T):
        p_data = IdentityPose.clone().unsqueeze(0).repeat(B, 1).to(device)
        # Translation is stored in indices 9, 10, 11 (last 3 of flattened 3x4)
        p_data[..., 9] = t * 0.05   # x: slight lateral drift
        p_data[..., 10] = 1.6       # y: head height ~1.6m
        p_data[..., 11] = t * 0.05  # z: forward motion 0.05m/frame
        poses.append(p_data)
    Ts_sr = PoseTW(torch.stack(poses, dim=1))  # B x T x 12
    batch[ARIA_IMG_T_SNIPPET_RIG[0]] = Ts_sr
    
    # --- Semi-dense points in world frame: B x T x N x 3 ---
    # Simulate points scattered in a room-like volume
    points = torch.zeros(B, T, n_points, 3, device=device)
    for t in range(T):
        # Random points in a 4m x 4m x 4m volume centered at (0, 2, 0)
        pts = torch.rand(B, n_points, 3, device=device) * 4.0 - 2.0
        pts[..., 1] += 2.0  # shift Y to [0, 4] (gravity-aligned height)
        points[:, t] = pts
    batch[ARIA_POINTS_WORLD] = points
    
    return batch

batch = create_synthetic_batch(B=B, T=T, device="cpu")

print("\n=== Batch Contents ===")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<35} shape={list(v.shape):<25} dtype={v.dtype}")
    else:
        print(f"  {k:<35} type={type(v).__name__:<15} shape={list(v.shape)}")

## 4. Stage-by-Stage Forward Pass

Let's trace through each processing stage to see what happens to the tensors.

### Stage 1: DINOv2 2D Feature Extraction

The frozen DINOv2 ViT-B/14 processes each RGB frame independently, producing 768-dim feature tokens at 1/14 spatial resolution. With `multilayer_output=True`, it returns features from 4 intermediate layers (3, 6, 9, 12) for the DPT head.

In [ ]:
# Move model and batch to device
model = model.to(device)
batch_gpu = {}
for k, v in batch.items():
    if hasattr(v, 'to'):
        batch_gpu[k] = v.to(device)
    else:
        batch_gpu[k] = v

print("=== Stage 1: DINOv2 2D Backbone ===")
print(f"Input: RGB images {list(batch_gpu[ARIA_IMG[0]].shape)}")
print(f"  -> B={B}, T={T}, C=3, H={RGB_H}, W={RGB_W}")

with torch.no_grad():
    backbone2d_out = model.backbone2d(batch_gpu)

rgb_feats = backbone2d_out["rgb"]
if isinstance(rgb_feats, list):
    print(f"\nOutput: {len(rgb_feats)} feature layers (for DPT multi-scale fusion):")
    for i, feat in enumerate(rgb_feats):
        print(f"  Layer {i}: {list(feat.shape)}")
    print(f"\nSpatial resolution: {RGB_H}x{RGB_W} -> {rgb_feats[0].shape[-2]}x{rgb_feats[0].shape[-1]} (1/{RGB_H // rgb_feats[0].shape[-2]}x downscale)")
    print(f"Feature dim: {rgb_feats[0].shape[2]}")
else:
    print(f"\nOutput: {list(rgb_feats.shape)}")

# Show GPU memory
if torch.cuda.is_available():
    print(f"\nGPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Stage 2: Lifter -- 2D Features to 3D Voxel Grid

This is the **core architectural contribution**. The Lifter:
1. Creates a gravity-aligned 3D voxel grid (using the last frame's pose)
2. Projects each voxel center into every camera view
3. Samples the 2D feature map at those projected locations
4. Averages features across all T frames
5. Concatenates a point-count mask and free-space mask

In [ ]:
# Add 2D features to batch (as the real pipeline does)
for stream in ["rgb"]:
    if stream in backbone2d_out:
        batch_gpu[f"{stream}/feat"] = backbone2d_out[stream]

print("=== Stage 2: Lifter (2D -> 3D) ===")
lifter = model.backbone3d

vD, vH, vW = lifter.voxel_size
extent = lifter.voxel_extent
print(f"Voxel grid: {vD}x{vH}x{vW} = {vD*vH*vW:,} voxels")
print(f"Voxel extent: x=[{extent[0]}, {extent[1]}], y=[{extent[2]}, {extent[3]}], z=[{extent[4]}, {extent[5]}] meters")
print(f"Voxel resolution: {lifter.voxel_meters:.4f} m/voxel")
print(f"Streams: {lifter.streams}")
print(f"Output dim: {lifter.output_dim()} = {lifter.out_dim}(features) + 1(point_mask) + 1(freespace)")

with torch.no_grad():
    lifter_out = lifter(batch_gpu)

print(f"\n--- Lifter Outputs ---")
for k, v in lifter_out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<35} {list(v.shape)}")
    elif hasattr(v, 'shape'):
        print(f"  {k:<35} {type(v).__name__} {list(v.shape)}")
    elif isinstance(v, list):
        print(f"  {k:<35} list of {len(v)} items")

if torch.cuda.is_available():
    print(f"\nGPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Stage 3: 3D Neck (InvResNet-FPN)

The lifted voxel features go through a 3D encoder-decoder with FPN skip connections. This is a 3D analog of a 2D feature pyramid network.

In [ ]:
print("=== Stage 3: 3D Neck (InvResNet-FPN) ===")
voxel_feats = lifter_out["voxel/feat"]
print(f"Input: {list(voxel_feats.shape)} = B x C x D x H x W")

with torch.no_grad():
    neck_feats = model.neck(voxel_feats)

print(f"Output: {list(neck_feats.shape)}")
print(f"\nThe neck compresses {voxel_feats.shape[1]} channels -> {neck_feats.shape[1]} channels")
print(f"while maintaining spatial resolution at {neck_feats.shape[2]}x{neck_feats.shape[3]}x{neck_feats.shape[4]}")

# Show the encoder path dimensions
print(f"\nEncoder path (InvResNet blocks):")
print(f"  Block1: {voxel_feats.shape[1]} -> 64  (stride=1, same resolution)")
print(f"  Block2: 64 -> 96   (stride=2, resolution /2)")
print(f"  Block3: 96 -> 128  (stride=2, resolution /4)")
print(f"  Block4: 128 -> 160 (stride=2, resolution /8)")
print(f"Decoder path (FPN upsampling):")
print(f"  FPN3: 160->128 + skip from Block3")
print(f"  FPN2: 128->96  + skip from Block2")
print(f"  FPN1: 96->64   + skip from Block1")

if torch.cuda.is_available():
    print(f"\nGPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Stage 4: Task Heads

Four parallel heads process the neck features:
- **Occupancy head** -> 1 channel -> sigmoid -> surface probability per voxel
- **Centerness head** -> 1 channel -> sigmoid -> object center probability per voxel
- **BBox head** -> 7 channels -> (h, w, d, offset_h, offset_w, offset_d, yaw)
- **Class head** -> 29 channels -> softmax -> class probability per voxel

In [ ]:
print("=== Stage 4: Task Heads ===")

with torch.no_grad():
    # Occupancy
    occ_logits = model.occ_head(neck_feats)
    occ_pr = torch.sigmoid(occ_logits)
    print(f"Occupancy head: {list(neck_feats.shape)} -> {list(occ_logits.shape)} -> sigmoid -> surface probability")
    
    # Centerness
    cent_logits = model.cent_head(neck_feats)
    cent_pr = torch.sigmoid(cent_logits)
    print(f"Centerness head: {list(neck_feats.shape)} -> {list(cent_logits.shape)} -> sigmoid -> center probability")
    
    # BBox
    bbox_pr = model.bbox_head(neck_feats)
    print(f"BBox head: {list(neck_feats.shape)} -> {list(bbox_pr.shape)} = [h, w, d, off_h, off_w, off_d, yaw]")
    
    # Class
    clas_logits = model.clas_head(neck_feats)
    clas_pr = torch.nn.functional.softmax(clas_logits, dim=1)
    print(f"Class head: {list(neck_feats.shape)} -> {list(clas_logits.shape)} -> softmax -> {model.num_class} classes")

print(f"\n--- Head Output Ranges (random weights, no real meaning yet) ---")
print(f"  Occupancy: [{occ_pr.min():.3f}, {occ_pr.max():.3f}] (should be ~0.5 with random weights)")
print(f"  Centerness: [{cent_pr.min():.6f}, {cent_pr.max():.6f}] (biased to ~0 by init bias=-5)")
print(f"  BBox sizes: [{bbox_pr[:,:3].min():.3f}, {bbox_pr[:,:3].max():.3f}] meters")
print(f"  Class max prob: {clas_pr.max():.3f}")

### Full Forward Pass

Now let's run the complete model end-to-end and see all outputs.

In [ ]:
# Fresh batch for clean forward pass
batch_full = create_synthetic_batch(B=1, T=10, device=device)

print("=== Full Forward Pass ===")
with torch.no_grad():
    out = model(batch_full)

print(f"\n--- All Model Outputs ---")
for k in sorted(out.keys()):
    v = out[k]
    if isinstance(v, torch.Tensor):
        print(f"  {k:<40} {str(list(v.shape)):<30} {v.dtype}")
    elif hasattr(v, 'shape'):
        print(f"  {k:<40} {type(v).__name__} {list(v.shape)}")
    elif isinstance(v, dict):
        print(f"  {k:<40} dict with {len(v)} keys")
    elif isinstance(v, list):
        print(f"  {k:<40} list of {len(v)} items")
    else:
        print(f"  {k:<40} {type(v).__name__}")

if torch.cuda.is_available():
    print(f"\nPeak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    torch.cuda.reset_peak_memory_stats()

## 5. Visualizations

In [ ]:
# Visualize occupancy predictions (slice through the voxel grid)
occ = out["occ_pr"][0, 0].cpu().numpy()  # D x H x W

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# XY slice (top-down view) at mid-height
mid_d = occ.shape[0] // 2
axes[0].imshow(occ[mid_d], cmap='hot', vmin=0, vmax=1)
axes[0].set_title(f'Occupancy -- Top-Down (D={mid_d})')
axes[0].set_xlabel('W (x-axis)')
axes[0].set_ylabel('H (y-axis)')

# XZ slice (side view) at mid-width
mid_h = occ.shape[1] // 2
axes[1].imshow(occ[:, mid_h, :], cmap='hot', vmin=0, vmax=1)
axes[1].set_title(f'Occupancy -- Side View (H={mid_h})')
axes[1].set_xlabel('W (x-axis)')
axes[1].set_ylabel('D (z-axis)')

# YZ slice (front view) at mid-depth
mid_w = occ.shape[2] // 2
axes[2].imshow(occ[:, :, mid_w], cmap='hot', vmin=0, vmax=1)
axes[2].set_title(f'Occupancy -- Front View (W={mid_w})')
axes[2].set_xlabel('H (y-axis)')
axes[2].set_ylabel('D (z-axis)')

for ax in axes:
    ax.set_aspect('equal')
plt.colorbar(axes[0].images[0], ax=axes, shrink=0.6, label='Occupancy Probability')
plt.suptitle('Occupancy Predictions (random weights -- patterns show network structure, not real geometry)', y=1.02)
plt.tight_layout()
plt.savefig('demo_occupancy_slices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: demo_occupancy_slices.png")

In [ ]:
# Visualize centerness predictions
cent = out["cent_pr"][0, 0].cpu().numpy()  # D x H x W

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

mid_d = cent.shape[0] // 2
im0 = axes[0].imshow(cent[mid_d], cmap='viridis')
axes[0].set_title(f'Centerness -- Top-Down (D={mid_d})')

mid_h = cent.shape[1] // 2
axes[1].imshow(cent[:, mid_h, :], cmap='viridis')
axes[1].set_title(f'Centerness -- Side View (H={mid_h})')

mid_w = cent.shape[2] // 2
axes[2].imshow(cent[:, :, mid_w], cmap='viridis')
axes[2].set_title(f'Centerness -- Front View (W={mid_w})')

for ax in axes:
    ax.set_aspect('equal')
plt.colorbar(im0, ax=axes, shrink=0.6, label='Centerness Score')
plt.suptitle('Centerness Predictions (note: bias=-5 init makes values very small)', y=1.02)
plt.tight_layout()
plt.savefig('demo_centerness_slices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Centerness range: [{cent.min():.6f}, {cent.max():.6f}]")
print("Saved: demo_centerness_slices.png")

In [ ]:
# Visualize the voxel grid feature channels (PCA of 3D features)
voxel_feat = lifter_out["voxel/feat"][0].cpu()  # C x D x H x W
C, D, H, W = voxel_feat.shape

# Reshape to (D*H*W) x C for PCA
feats_flat = voxel_feat.reshape(C, -1).T  # N x C

# Simple PCA via SVD (take top 3 components for RGB visualization)
feats_centered = feats_flat - feats_flat.mean(0)
U, S, Vh = torch.linalg.svd(feats_centered, full_matrices=False)
pca_3 = (feats_centered @ Vh[:3].T)  # N x 3

# Normalize to [0, 1] for visualization
pca_3 = pca_3 - pca_3.min(0).values
pca_3 = pca_3 / (pca_3.max(0).values + 1e-8)
pca_3 = pca_3.reshape(D, H, W, 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(pca_3[D//2].numpy())
axes[0].set_title('Voxel Features PCA -- Top-Down')

axes[1].imshow(pca_3[:, H//2, :].numpy())
axes[1].set_title('Voxel Features PCA -- Side View')

axes[2].imshow(pca_3[:, :, W//2].numpy())
axes[2].set_title('Voxel Features PCA -- Front View')

for ax in axes:
    ax.set_aspect('equal')
plt.suptitle(f'Lifted 3D Feature Volume (PCA of {C} channels -> RGB)', y=1.02)
plt.tight_layout()
plt.savefig('demo_voxel_features_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: demo_voxel_features_pca.png")

In [ ]:
# Visualize the point count mask and freespace mask
point_mask = lifter_out["voxel/occ_input"][0, 0].cpu().numpy()  # D x H x W

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(point_mask[D//2], cmap='Blues')
axes[0].set_title(f'Point Mask -- Top-Down (D={D//2})')

axes[1].imshow(point_mask[:, H//2, :], cmap='Blues')
axes[1].set_title(f'Point Mask -- Side View')

axes[2].imshow(point_mask[:, :, W//2], cmap='Blues')
axes[2].set_title(f'Point Mask -- Front View')

occupied = (point_mask > 0).sum()
total = point_mask.size
for ax in axes:
    ax.set_aspect('equal')
plt.suptitle(f'Semi-Dense Point Voxelization ({occupied}/{total} = {occupied/total*100:.1f}% occupied)', y=1.02)
plt.tight_layout()
plt.savefig('demo_point_mask.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: demo_point_mask.png")

## 6. Understanding the Voxel Grid Geometry

The voxel grid is **gravity-aligned** -- the Y axis always points up regardless of camera orientation. This is crucial for detecting objects with correct orientations.

In [ ]:
from efm3d.utils.voxel import create_voxel_grid

extent = lifter.voxel_extent
vD, vH, vW = lifter.voxel_size

print("=== Voxel Grid Geometry ===")
print(f"Grid dimensions: {vW} x {vH} x {vD} (W x H x D)")
print(f"Physical extent:")
print(f"  X (width):  [{extent[0]:+.1f}, {extent[1]:+.1f}] m  ({extent[1]-extent[0]:.1f}m total)")
print(f"  Y (height): [{extent[2]:+.1f}, {extent[3]:+.1f}] m  ({extent[3]-extent[2]:.1f}m total)")
print(f"  Z (depth):  [{extent[4]:+.1f}, {extent[5]:+.1f}] m  ({extent[5]-extent[4]:.1f}m total)")
print(f"Voxel size: {lifter.voxel_meters:.4f} m = {lifter.voxel_meters*100:.2f} cm")
print(f"Total volume: {(extent[1]-extent[0])*(extent[3]-extent[2])*(extent[5]-extent[4]):.0f} m^3")

# Create the actual voxel grid to show positions
vox_v = create_voxel_grid(vW, vH, vD, extent, device="cpu")
print(f"\nVoxel center positions tensor: {list(vox_v.shape)} = W x H x D x 3")
print(f"Corner voxel positions:")
print(f"  [0,0,0]: ({vox_v[0,0,0,0]:.3f}, {vox_v[0,0,0,1]:.3f}, {vox_v[0,0,0,2]:.3f})")
print(f"  [-1,-1,-1]: ({vox_v[-1,-1,-1,0]:.3f}, {vox_v[-1,-1,-1,1]:.3f}, {vox_v[-1,-1,-1,2]:.3f})")

# The T_world_voxel pose
T_wv = lifter_out.get("voxel/T_world_voxel")
if T_wv is not None:
    print(f"\nT_world_voxel (gravity-aligned pose): {type(T_wv).__name__} {list(T_wv.shape)}")
    print(f"  Translation: ({T_wv.t[0, 0]:.3f}, {T_wv.t[0, 1]:.3f}, {T_wv.t[0, 2]:.3f})")

## 7. OBB Post-Processing Pipeline

The raw dense voxel predictions get converted to sparse oriented bounding boxes via:
1. **Simple NMS** on the centerness heatmap (suppress non-maxima in a local neighborhood)
2. **Top-K selection** of highest centerness voxels
3. **Voxel-to-OBB conversion** (decode bbox size + offset + yaw from selected voxels)
4. **Temporal tracking** (optional, IoU-based association across snippets)

In [ ]:
from efm3d.utils.detection_utils import simple_nms3d, voxel2obb

print("=== OBB Post-Processing ===")

cent_pr = out["cent_pr"]
bbox_pr = out["bbox_pr"]
clas_pr = out["clas_pr"]

# Step 1: NMS on centerness heatmap
with torch.no_grad():
    cent_pr_nms = simple_nms3d(cent_pr, nms_radius=model.splat_sigma + 1)

n_before = (cent_pr > model.det_thresh).sum().item()
n_after = (cent_pr_nms > model.det_thresh).sum().item()
print(f"Step 1 -- Simple NMS (radius={model.splat_sigma + 1}):")
print(f"  Voxels above threshold ({model.det_thresh}): {n_before} -> {n_after}")

# Step 2: Voxel to OBB conversion
with torch.no_grad():
    obbs_pr, _, clas_prob = voxel2obb(
        cent_pr_nms, bbox_pr, clas_pr,
        model.ve, top_k=128, thresh=model.det_thresh,
        return_full_prob=True,
    )

if obbs_pr is not None:
    print(f"\nStep 2 -- Voxel-to-OBB conversion:")
    print(f"  Detected {len(obbs_pr)} OBBs (top-128 filtered by threshold)")
    print(f"  OBB tensor: {type(obbs_pr).__name__} shape={list(obbs_pr.shape)}")
else:
    print(f"\nNo OBBs detected above threshold (expected with random weights)")

print(f"\nDetection parameters:")
print(f"  det_thresh: {model.det_thresh}")
print(f"  bbox_min/max: [{model.bbox_min}, {model.bbox_max}] meters")
print(f"  offset_max: {model.offset_max:.4f} meters")
print(f"  yaw_max: {model.yaw_max} radians")
print(f"  iou_threshold: {model.iou_thres}")
print(f"  num_classes: {model.num_class}")

## 8. Real Inference (if checkpoint available)

If you have downloaded the model checkpoint from [Project Aria](https://www.projectaria.com/research/efm3D/), this section runs real inference.

In [ ]:
import os

CKPT_SERVER = "./ckpt/model_release.pth"
CKPT_DESKTOP = "./ckpt/model_lite.pth"
SAMPLE_DATA = "./data/seq136_sample/video.vrs"

has_ckpt = os.path.exists(CKPT_SERVER) or os.path.exists(CKPT_DESKTOP)
has_data = os.path.exists(SAMPLE_DATA)

print("=== Checkpoint & Data Status ===")
print(f"  Server checkpoint (model_release.pth):  {'FOUND' if os.path.exists(CKPT_SERVER) else 'Not found'}")
print(f"  Desktop checkpoint (model_lite.pth):    {'FOUND' if os.path.exists(CKPT_DESKTOP) else 'Not found'}")
print(f"  Sample data (seq136_sample/video.vrs):  {'FOUND' if has_data else 'Not found'}")

if not has_ckpt:
    print("\n" + "="*60)
    print("TO GET THE CHECKPOINT:")
    print("  1. Go to https://www.projectaria.com/research/efm3D/")
    print("  2. Enter your email to get download access")
    print("  3. Download evl_model_ckpt.zip")
    print("  4. Place it in ./ckpt/ and run:")
    print("     sh prepare_inference.sh")
    print("="*60)

if has_ckpt and has_data:
    print("\nReady to run real inference! Proceeding...")

In [ ]:
# Run real inference if checkpoint and data are available
if has_ckpt and has_data:
    ckpt_path = CKPT_DESKTOP if os.path.exists(CKPT_DESKTOP) else CKPT_SERVER
    cfg_path = "./efm3d/config/evl_inf_desktop.yaml" if "lite" in ckpt_path else "./efm3d/config/evl_inf.yaml"
    
    print(f"Loading checkpoint: {ckpt_path}")
    print(f"Using config: {cfg_path}")
    
    from efm3d.inference.pipeline import run_one
    
    run_one(
        data_path=SAMPLE_DATA,
        model_ckpt=ckpt_path,
        model_cfg=cfg_path,
        max_snip=5,  # only process 5 snippets for quick demo
        snip_stride=0.5,
        voxel_res=0.08,
        output_dir="./output_demo",
        skip_video=True,
    )
    print("\nInference complete! Check ./output_demo/ for results.")
else:
    print("Skipping real inference (no checkpoint/data). See instructions above.")

## 9. Inference Pipeline Deep-Dive

Even without real data, let's understand the full inference pipeline structure.

In [ ]:
print("""
=== Full Inference Pipeline ===

1. DATA LOADING
   |-- VrsSequenceDataset: reads .vrs (Aria native format)
   |   +-- Extracts: RGB frames, SLAM frames, poses, calibs, semi-dense points
   +-- AtekWdsStreamDataset: reads WebDataset .tar shards (ATEK format)
       +-- Pre-processed data in standard format

2. PER-SNIPPET INFERENCE (EfmInference)
   |-- For each 1-second snippet (10 frames):
   |   |-- Run EVL model -> occupancy grid + OBB predictions
   |   |-- Save per-snippet occupancy to disk
   |   +-- Collect OBB predictions
   +-- Snippet stride controls overlap (default 0.1s = 90% overlap)

3. OBB TRACKING (track_obbs)
   |-- IoU-based temporal association across snippets
   |-- Instantiation: require object seen in N snippets
   +-- Output: tracked_scene_obbs.csv

4. VOLUMETRIC FUSION (VolumetricFusion)
   |-- TSDF fusion from per-snippet occupancy grids
   |-- Voxel resolution configurable (default 4cm, can use 8cm for speed)
   |-- Marching cubes -> triangle mesh
   +-- Output: fused_mesh.ply

5. EVALUATION
   |-- OBB: mAP at IoU=0.2 (from tracked_scene_obbs.csv vs gt)
   +-- Mesh: accuracy, completeness, precision, recall (vs gt mesh)

6. VISUALIZATION
   +-- Generate video overlay with OBBs + mesh
""")

## 10. Key Design Decisions Explained

In [ ]:
print("""
=== Why These Design Choices? ===

1. GRAVITY-ALIGNED VOXEL GRID
   - Y-axis always points up -> objects have consistent orientation
   - OBB yaw angle is relative to gravity (not camera)
   - Computed from camera pose + known gravity direction (VIO)

2. FROZEN DINOv2 BACKBONE
   - 86M params that don't need gradients -> saves GPU memory
   - Self-supervised features transfer well to 3D tasks
   - Only ~50M trainable params in the 3D components

3. DPT HEAD (not simple CNN upsampling)
   - Multi-scale feature fusion from 4 ViT layers
   - Better spatial detail preservation for the lifting step
   - Upsamples 1/14 resolution -> original resolution

4. EXPLICIT VOXEL GRID (not implicit/NeRF)
   - Interpretable 3D representation
   - Easy to add task-specific heads
   - Efficient inference (single forward pass per snippet)
   - Supports volumetric fusion across snippets

5. SEMI-DENSE POINTS AS INPUT
   - Aria provides sparse 3D points from VIO (not dense depth)
   - Point mask tells the model where surfaces exist
   - Freespace mask tells the model where space is empty
   - These geometric cues complement the appearance features

6. CENTERNESS-BASED DETECTION (not anchor-based)
   - Anchor-free: each voxel predicts centerness + box params
   - Similar to CenterNet/FCOS but in 3D
   - Simpler than anchor-based approaches, works well for indoor scenes

7. SYNTHETIC TRAINING (ASE) -> REAL EVAL (ADT, AEO)
   - Synthetic data provides unlimited perfect annotations
   - Domain gap bridged by: DINOv2 features + physical fidelity of ASE
   - This is a key finding: sim->real transfer works for egocentric 3D
""")

## 11. Memory & Performance Profile

In [ ]:
import time

# Profile forward pass timing
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

batch_perf = create_synthetic_batch(B=1, T=10, device=device)

# Warmup
with torch.no_grad():
    _ = model(batch_perf)
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Timed run
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
with torch.no_grad():
    _ = model(batch_perf)
if torch.cuda.is_available():
    torch.cuda.synchronize()
t1 = time.time()

print("=== Performance Profile (Desktop Config, 48^3 voxels) ===")
print(f"Forward pass time: {(t1-t0)*1000:.0f} ms")
if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
    print(f"Current GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

print(f"\nFor a 60-second Aria sequence at 0.1s stride:")
n_snippets = int(60 / 0.1)
est_time = n_snippets * (t1-t0)
print(f"  ~{n_snippets} snippets x {(t1-t0)*1000:.0f}ms = ~{est_time:.0f}s total")

## 12. Cleanup

In [ ]:
# Free GPU memory
del model, batch_gpu, batch_full, batch_perf, out, lifter_out, backbone2d_out
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e6:.0f} MB")

print("\n=== Summary ===")
print("This notebook demonstrated:")
print("  1. EVL model construction from Hydra configs")
print("  2. Parameter counting by component")
print("  3. Synthetic Aria data batch creation")
print("  4. Stage-by-stage forward pass with shape tracing")
print("  5. Visualization of occupancy, centerness, voxel features")
print("  6. OBB post-processing pipeline")
print("  7. Performance profiling")
print("\nNext steps:")
print("  * Download the model checkpoint for real inference")
print("  * Download ASE eval data to test on synthetic sequences")
print("  * Explore training with: python train.py")